# IV Curve Analysis with Gaussian Filtering

This notebook analyzes IV characteristic data from TMR measurements using Gaussian filtering.

Features:
1. Gaussian filtering with sigma = 1.5
2. Voltage offset detection (zero-crossing)
3. Residual RMS quality metric
4. Comprehensive visualization

In [ ]:
# Notebook setup - enables autoreload, imports, and path management
from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Import Gaussian filter modules
from scripts.IV_Hscan_gaussian import (
    load_iv_data,
    data_parser_IV,
    filter_iv_data,
    save_filter_plots
)

## Load and Parse Data

In [ ]:
# File path (relative to project root)
file_path = PROJECT_ROOT / r"data/device 2/b_scans/IV_H_scans/20251206064739b_scan_70K__62steps_H3.00T_phi_84.0to84.0_theta_0.0to0.0°_v0.datfolder/20251206070129_Hx_94.050000E-3_T_Hy_-210.000000E-6_T_Hz_895.000000E-3_T_decreasing.txt.dat"


print("Loading data...")
iv_data = load_iv_data(str(file_path))
extended_data = data_parser_IV(iv_data)

# Display metadata
print(f"\nMetadata:")
print(f"Timestamp: {extended_data.timestamp}")
print(f"H (signed) = {extended_data.H:.6e} T")
print(f"Hx = {extended_data.Hx:.6e} T")
print(f"Hy = {extended_data.Hy:.6e} T")
print(f"Hz = {extended_data.Hz:.6e} T")

## Plot Raw IV Curve

In [ ]:
# Plot raw V-I curve
plt.figure(figsize=(10, 6))
plt.plot(extended_data.voltage, np.abs(extended_data.current - 3E-10) * 1e6, 'o-', 
         linewidth=2, markersize=4, label='Raw Data')
plt.xlabel('Voltage (V)', fontsize=14)
plt.ylabel('Current (µA)', fontsize=14)
plt.title(f'Raw IV Curve at H = {extended_data.H:.4f} T\n' + 
          f'Hx={extended_data.Hx:.3e} T, Hy={extended_data.Hy:.3e} T, Hz={extended_data.Hz:.3e} T',
          fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.yscale('log')


#plt.xlim(-0.2, 0.2)
#plt.ylim(-0.002, 0.002)
plt.show()

## Apply Gaussian Filter

We apply a Gaussian filter with σ = 1.5 to smooth the IV curve and estimate the voltage offset where the filtered current crosses zero.

In [ ]:
# Apply Gaussian filter
print("Applying Gaussian filter...")
filtered_data = filter_iv_data(extended_data, sigma=1.5, remove_outliers_flag=False)

# Display results
print("\n=== Gaussian Filter Results ===")
print(f"H = {filtered_data.H:.6e} T")
print(f"Sigma = {filtered_data.sigma:.2f}")
print(f"V_offset = {filtered_data.v_offset:.6f} V")
print(f"Residual RMS = {filtered_data.residual_rms:.6e} A")
print(f"Residual RMS = {filtered_data.residual_rms * 1e6:.3f} µA")

## Visualize Filtered Results

In [ ]:
# Create custom plot for notebook visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top plot: Raw vs Filtered
ax1.plot(filtered_data.voltage, filtered_data.current * 1e6, 'o',
         markersize=4, label='Raw Data', alpha=0.5, color='blue')
ax1.plot(filtered_data.voltage_smooth, filtered_data.current_smooth * 1e6, '-',
         linewidth=2.5, label=f'Gaussian Filter (σ={filtered_data.sigma})', color='red')

# Mark voltage offset
if not np.isnan(filtered_data.v_offset):
    ax1.axvline(filtered_data.v_offset, color='green', linestyle='--',
                linewidth=2, label=f'V_offset = {filtered_data.v_offset:.4f} V')
    ax1.axhline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.3)

ax1.set_xlabel('Voltage (V)', fontsize=12)
ax1.set_ylabel('Current (µA)', fontsize=12)
ax1.set_title(
    f'Gaussian Filtered IV Curve (H = {filtered_data.H:.4f} T)\n'
    f'σ = {filtered_data.sigma:.2f}, V_offset = {filtered_data.v_offset:.4f} V',
    fontsize=12
)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Bottom plot: Residuals
residuals = filtered_data.current_clean - filtered_data.current_filtered
ax2.plot(filtered_data.voltage_clean, residuals * 1e6, 'o-',
         markersize=4, linewidth=1, color='purple', alpha=0.6)
ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)

ax2.set_xlabel('Voltage (V)', fontsize=12)
ax2.set_ylabel('Residual (µA)', fontsize=12)
ax2.set_title(
    f'Residuals (Raw - Filtered)\n'
    f'RMS = {filtered_data.residual_rms * 1e6:.3f} µA',
    fontsize=12
)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6)) 
plt.plot(filtered_data.voltage_smooth, filtered_data.current_smooth * 1e6, '-',
         linewidth=2.5, label=f'Gaussian Filter (σ={filtered_data.sigma })', color='red', alpha=1)
plt.plot(extended_data.voltage, extended_data.current * 1e6, 'o',
         markersize=4, label='Raw Data', alpha=0.2, color='blue')
plt.xlabel('Voltage (V)', fontsize=14)  
plt.ylabel('Current (µA)', fontsize=14)
plt.title(f'IV Curve at H = {extended_data.H:.4f} T\n' + 
          f'Hx={extended_data.Hx:.3e} T, Hy={extended_data.Hy:.3e} T, Hz={extended_data.Hz:.3e} T',
          fontsize=14)
plt.legend()
plt.xlim(-0.2, 0.2)
plt.ylim(-0.001, 0.001)


## Symmetric and Asymmetric Components

The Gaussian filter analysis automatically decomposes the filtered current into:
- **Symmetric component**: I_sym(V) = [I(V) + I(-V)] / 2  (even function)
- **Asymmetric component**: I_asym(V) = [I(V) - I(-V)] / 2  (odd function)

These components are pre-calculated and available in `filtered_data`.

In [ ]:
# Plot Symmetric and Asymmetric Components (I vs V)
# Components are pre-calculated in filter_iv_data()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# --- PLOT 1: Symmetric Components ---
ax1.set_title(f'Symmetric (Even) Component at H = {filtered_data.H:.4f} T', fontsize=14)

# Plot the raw symmetric component as scattered points
ax1.plot(filtered_data.voltage, filtered_data.current_filtered_sym * 1e6, 
         'o', color='gray', markersize=4, alpha=0.5, label='Symmetric (at measured V)')
# Plot the smoothed symmetric component as a clean line
ax1.plot(filtered_data.voltage_smooth, filtered_data.current_smooth_sym * 1e6, 
         '-', color='dodgerblue', linewidth=2.5, label='Symmetric (smooth)')

ax1.set_xlabel('Voltage (V)', fontsize=12)
ax1.set_ylabel('Symmetric Current (µA)', fontsize=12)
ax1.grid(True, alpha=0.4)
ax1.legend()

# --- PLOT 2: Asymmetric Components ---
ax2.set_title(f'Asymmetric (Odd) Component at H = {filtered_data.H:.4f} T', fontsize=14)

# Plot the raw asymmetric component
ax2.plot(filtered_data.voltage, filtered_data.current_filtered_asym * 1e6, 
         'o', color='gray', markersize=4, alpha=0.5, label='Asymmetric (at measured V)')
# Plot the smoothed asymmetric component
ax2.plot(filtered_data.voltage_smooth, filtered_data.current_smooth_asym * 1e6, 
         '-', color='orangered', linewidth=2.5, label='Asymmetric (smooth)')

ax2.set_xlabel('Voltage (V)', fontsize=12)
ax2.set_ylabel('Asymmetric Current (µA)', fontsize=12)
ax2.grid(True, alpha=0.4)
ax2.legend()

plt.tight_layout()
plt.show()

# Verify decomposition
I_reconstructed = filtered_data.current_filtered_sym + filtered_data.current_filtered_asym
reconstruction_error = np.max(np.abs(I_reconstructed - filtered_data.current_filtered))
print(f"\nDecomposition verification:")
print(f"Max reconstruction error: {reconstruction_error:.2e} A (should be ~0)")

In [ ]:
# Save plots
print("Saving plots...")
save_path = PROJECT_ROOT / "output/gaussian_filters"
saved_file = save_filter_plots(filtered_data, save_path)
print(f"Plots saved to: {saved_file}")

## Compare Raw vs Filtered at Specific Voltages

Extract current values at specific voltages to see the effect of filtering.

In [ ]:
# Define voltages of interest
voltages_of_interest = [-0.4, -0.2, 0.0, 0.2, 0.4]

print("\n=== Current Values at Specific Voltages ===")
print(f"{'Voltage (V)':<12} {'I_raw (µA)':<15} {'I_filtered (µA)':<15} {'Difference (µA)':<15}")
print("-" * 60)

for V_target in voltages_of_interest:
    # Find closest voltage in raw data
    idx = np.argmin(np.abs(filtered_data.voltage - V_target))
    V_actual = filtered_data.voltage[idx]
    I_raw = filtered_data.current[idx] * 1e6
    I_filt = filtered_data.current_filtered[idx] * 1e6
    diff = I_raw - I_filt
    
    print(f"{V_actual:<12.4f} {I_raw:<15.4f} {I_filt:<15.4f} {diff:<15.4f}")

## Summary

This notebook demonstrated:
1. Loading IV data from .dat files
2. Applying Gaussian filter (σ=1.5)
3. Detecting voltage offset from zero-crossing
4. Calculating residual RMS as a quality metric
5. Visualizing raw vs filtered data

For batch processing of multiple files, see `IV_H_scan_gaussian_batch_analysis.ipynb`.